# Token Communications (TokCom) Demo

**Citation:** L. Qiao, M. B. Mashhadi, Z. Gao, R. Tafazolli, M. Bennis and D. Niyato, "Token Communications: A Large Model-Driven Framework for Cross-Modal Context-Aware Semantic Communications," in IEEE Wireless Communications, vol. 32, no. 5, pp. 80-88, October 2025, doi: 10.1109/MWC.001.2500084.

**Keywords:** Token networks, Large language models, Context awareness, Semantic communication, Transformers

---

This notebook demonstrates the TokCom framework. This demo leverages VQGAN codebook and MaskGiT for efficient token-based semantic communication in wireless networks. The framework enables robust image transmission through token-level generative error correction. Cross-modality information is used, like class label, to further improve the performance. 


## Overview

The TokCom framework consists of three main components:
1. **Token Encoding**: Convert images to discrete tokens using VQGAN
2. **Wireless Transmission**: Simulate packet-based transmission with error correction
3. **Generative Reconstruction**: Use MaskGiT to reconstruct images from corrupted tokens

This demo evaluates the system performance across different SNR levels and compares three reconstruction approaches:
- **TokCom**: Our proposed generative reconstruction method
- **Direct Decoding**: Direct VQGAN decoding of received tokens
- **Reference**: Perfect reconstruction without transmission errors


## Setup and Dependencies


In [ ]:
# Core dependencies
import os
import random
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import argparse

# Model dependencies
from Trainer.vit import MaskGIT
from Performance_metrics import calculate_psnr, calculate_lpips, calculate_clip_score
import lpips
import clip

# Configure TensorFlow GPU memory growth before importing Sionna
import tensorflow as tf
# Set TensorFlow to use memory growth to avoid allocating all GPU memory
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("TensorFlow GPU memory growth enabled")
    except RuntimeError as e:
        print(f"TensorFlow GPU configuration error: {e}")

# Wireless communication dependencies
import sionna
from sionna.fec.crc import CRCEncoder, CRCDecoder
from sionna.fec.conv import ConvEncoder, ViterbiDecoder
from sionna.mapping import Mapper, Demapper

print("Dependencies loaded successfully!")


## Environment

- OS: Ubuntu (Linux kernel)
- GPUs: 2 x NVIDIA GeForce RTX 2080 Ti
- System memory: 24 GB
- CUDA: Auto-detected via PyTorch/TensorFlow

Notes:
- TensorFlow GPU memory growth is enabled to avoid pre-allocating all GPU memory.
- On multi-GPU systems, you can restrict TensorFlow to a specific GPU:
```python
# Keep only GPU:0 visible to TensorFlow
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices(gpus[0], 'GPU')
```
- To run Sionna on CPU (leaving GPUs fully to PyTorch), before importing TensorFlow:
```python
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import tensorflow as tf
```


## Configuration and Model Initialization


In [ ]:
# Configuration class
class Config:
    def __init__(self):
        # Model paths
        self.vit_folder = "./pretrained_maskgit/MaskGIT/MaskGIT_ImageNet_256.pth"
        self.vqgan_folder = "./pretrained_maskgit/VQGAN/"

        # Image parameters
        self.img_size = 256
        self.patch_size = self.img_size // 16  # 16x16 patches
        self.mask_value = 1024  # Masked token value
        
        # System parameters
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.seed = 1
        
        # Data parameters
        self.data_folder = os.environ.get("IMAGENET_ROOT", "/datasets_local/ImageNet/")
        self.num_images = 40
        self.class_indices = [503, 498]
        # [599, 503, 137, 936, 64, 74, 99, 498, 299, 268, 180, 765] # default 12 classes
        
        # Generation parameters
        self.sm_temp = 1.3      # Softmax temperature
        self.r_temp = 7         # Gumbel temperature
        self.step = 32          # Generation steps
        self.sched_mode = "arccos"  # Noise scheduler
        self.randomize = "linear"
        
        # Wireless parameters
        self.snr_list = [5, 7]
        # [5, 6, 7, 7.5, 8, 9, 10, 11]

        self.num_simulations = 1
        
        # Output directory
        self.save_dir = "./tokcom_results"
        os.makedirs(self.save_dir, exist_ok=True)

# Initialize configuration
config = Config()
print(f"Using device: {config.device}")

# Set random seeds for reproducibility
if config.seed > 0:
    torch.manual_seed(config.seed)
    torch.cuda.manual_seed(config.seed)
    np.random.seed(config.seed)
    random.seed(config.seed)
    torch.backends.cudnn.deterministic = True

# Initialize MaskGiT model
args = argparse.Namespace()
args.vit_folder = config.vit_folder
args.vqgan_folder = config.vqgan_folder
args.img_size = config.img_size
args.patch_size = config.patch_size
args.mask_value = config.mask_value
args.device = config.device
args.seed = config.seed
args.data = "imagenet"
args.resume = True
args.debug = True
# Additional parameters required by Trainer (not used when debug=True)
args.writer_log = ""
args.data_folder = config.data_folder
args.channel = 3
args.num_workers = 4
args.iter = 1_500_000
args.global_epoch = 380
args.lr = 1e-4
args.drop_label = 0.1
args.test_only = False
args.is_master = True
args.is_multi_gpus = False

maskgit = MaskGIT(args)
print("MaskGiT model loaded successfully!")


## Data Loading and Preprocessing


In [ ]:
def load_synset_mapping(file_path):
    """Load ImageNet synset mapping from file."""
    synset_dict = {}
    with open(file_path, "r", encoding="utf-8") as file:
        for idx, line in enumerate(file):
            key = line.split()[0]
            synset_dict[key] = idx
    return synset_dict

def select_images_from_classes(directory_path, class_indices, synset_dict, num_images=20, seed=None):
    """Select images from specified ImageNet classes."""
    if seed is not None:
        random.seed(seed)
    
    index_to_folder = {v: k for k, v in synset_dict.items()}
    all_image_paths = []
    all_class_labels = []
    
    for class_index in class_indices:
        if class_index not in index_to_folder:
            raise ValueError(f"Class {class_index} not found in synset mapping.")
        
        selected_folder = index_to_folder[class_index]
        folder_path = os.path.join(directory_path, selected_folder)
        
        images = [img for img in os.listdir(folder_path) 
                 if img.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if len(images) < num_images:
            raise ValueError(f"Not enough images in {selected_folder}.")
        
        selected_images = random.sample(images, num_images)
        selected_image_paths = [os.path.join(folder_path, img) for img in selected_images]
        
        class_label = synset_dict.get(selected_folder, None)
        if class_label is None:
            raise ValueError(f"Class label not found for {selected_folder}.")
        
        all_image_paths.append(selected_image_paths)
        all_class_labels.append(class_label)
    
    return all_image_paths, all_class_labels

def preprocess_image(img_path, transform):
    """Load and preprocess a single image."""
    img = Image.open(img_path)
    
    # Convert to RGB if needed
    if img.mode == 'L':
        img = img.convert('RGB')
    elif img.mode == 'LA':
        img = Image.merge("RGB", (img.getchannel('L'),) * 3)
    elif img.mode == 'RGBA':
        img = img.convert('RGB')
    
    return transform(img).unsqueeze(0)

# Load synset mapping
synset_dict = load_synset_mapping("synset_words.txt")

# Load image data
directory_path = '/root/autodl-tmp/imagenet100'
ImagePathAll, ClassLabelAll = select_images_from_classes(
    directory_path, config.class_indices, synset_dict, num_images=config.num_images)

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

print(f"Loaded {len(ImagePathAll)} classes with {config.num_images} images each")


## Wireless Communication Simulation


In [ ]:
class WirelessSimulation:
    """Simulate wireless transmission with error correction."""
    
    def __init__(self, toknum=256, toks_per_pkg=16, bits_per_tok=10, rate=0.5):
        self.toknum = toknum
        self.toks_per_pkg = toks_per_pkg
        self.bits_per_tok = bits_per_tok
        self.rate = rate
        
        # TensorFlow GPU memory is already set to growth mode in the imports cell
        # (tf.config.experimental.set_memory_growth). That prevents full GPU pre-allocation.
        # If you still see memory pressure, consider limiting visible devices like:
        # tf.config.set_visible_devices(gpus[0], 'GPU')  # keep only GPU:0 visible
        # or run Sionna on CPU by setting os.environ['CUDA_VISIBLE_DEVICES'] = '' before TF import.
        pass
        
        # Initialize error correction components
        self.crc_encoder = CRCEncoder("CRC11")
        self.crc_degree = 11
        self.crc_decoder = CRCDecoder(self.crc_encoder)
        self.conv_encoder = ConvEncoder(gen_poly=["101", "111"])
        self.conv_decoder = ViterbiDecoder(encoder=self.conv_encoder)
    
    @torch.no_grad()
    def tokens_to_bits(self, tokens):
        """Convert tokens to binary representation."""
        batch_size, seq_len = tokens.shape
        bit_masks = (2 ** torch.arange(self.bits_per_tok - 1, -1, -1, device=tokens.device)).reshape(-1, 1)
        tokens_expanded = tokens.unsqueeze(1)
        bits_3d = ((tokens_expanded & bit_masks) > 0).float()
        return bits_3d.permute(0, 2, 1)
    
    @torch.no_grad()
    def bits_to_tokens(self, bits):
        """Convert binary representation back to tokens."""
        batch_size, _, _ = bits.shape
        powers = 2 ** torch.arange(self.bits_per_tok - 1, -1, -1, device=bits.device, dtype=torch.float32)
        tokens = torch.matmul(bits, powers.reshape(-1, 1)).squeeze().long()
        return tokens
    
    @torch.no_grad()
    def tokens_to_pkgs_interleaved(self, tokens):
        """Package tokens using interleaved allocation strategy."""
        batch_size, num_tokens = tokens.shape
        num_packages = num_tokens // self.toks_per_pkg
        device = tokens.device
        
        # Create mapping table for interleaved allocation
        token_to_pkg_pos = torch.zeros(num_tokens, 2, dtype=torch.long, device=device)
        for token_idx in range(num_tokens):
            pkg_idx = token_idx % num_packages
            pos_in_pkg = token_idx // num_packages
            token_to_pkg_pos[token_idx, 0] = pkg_idx
            token_to_pkg_pos[token_idx, 1] = pos_in_pkg
        
        # Reorder tokens according to mapping
        reordered_tokens = torch.zeros_like(tokens)
        for b in range(batch_size):
            for token_idx in range(num_tokens):
                pkg_idx = token_to_pkg_pos[token_idx, 0]
                pos_in_pkg = token_to_pkg_pos[token_idx, 1]
                new_pos = pkg_idx * self.toks_per_pkg + pos_in_pkg
                reordered_tokens[b, new_pos] = tokens[b, token_idx]
        
        # Convert to bits and reshape
        bits = self.tokens_to_bits(reordered_tokens)
        bits_reshaped = bits.reshape(batch_size, -1, self.bits_per_tok * self.toks_per_pkg)
        
        # Store mapping for later use
        self.token_to_pkg_pos = token_to_pkg_pos
        return bits_reshaped
    
    @torch.no_grad()
    def pkgs_to_tokens_interleaved(self, pkgs):
        """Convert packages back to original token order."""
        batch_size, num_packages, _ = pkgs.shape
        num_tokens = self.toknum
        device = pkgs.device
        
        # Recreate mapping if not available
        if not hasattr(self, 'token_to_pkg_pos'):
            token_to_pkg_pos = torch.zeros(num_tokens, 2, dtype=torch.long, device=device)
            for token_idx in range(num_tokens):
                pkg_idx = token_idx % num_packages
                pos_in_pkg = token_idx // num_packages
                token_to_pkg_pos[token_idx, 0] = pkg_idx
                token_to_pkg_pos[token_idx, 1] = pos_in_pkg
            self.token_to_pkg_pos = token_to_pkg_pos
        
        # Convert packages to tokens
        bits = pkgs.reshape(batch_size, -1, self.bits_per_tok)
        reordered_tokens = self.bits_to_tokens(bits)
        
        # Restore original order
        original_tokens = torch.zeros_like(reordered_tokens)
        for b in range(batch_size):
            for token_idx in range(num_tokens):
                pkg_idx = self.token_to_pkg_pos[token_idx, 0]
                pos_in_pkg = self.token_to_pkg_pos[token_idx, 1]
                new_pos = pkg_idx * self.toks_per_pkg + pos_in_pkg
                original_tokens[b, token_idx] = reordered_tokens[b, new_pos]
        
        return original_tokens
    
    @torch.no_grad()
    def QAM_modulation(self, bits, Mqam):
        """QAM modulation using Sionna."""
        batch_size, num_tokens, bits_per_pkg = bits.shape
        bits_reshaped = bits.reshape(batch_size, -1, Mqam)
        
        bits_np = bits_reshaped.cpu().numpy()
        bits_tf = tf.convert_to_tensor(bits_np, dtype=tf.float32)
        
        mapper = Mapper("qam", num_bits_per_symbol=Mqam)
        symbols_tf = mapper(bits_tf)
        
        symbols_np = symbols_tf.numpy()
        symbols = torch.from_numpy(symbols_np).to(bits.device)
        
        return symbols.reshape(batch_size, -1)
    
    @torch.no_grad()
    def QAM_demodulation_llr(self, symbols, snr, Mqam, bits_per_pkg=None):
        """QAM demodulation with LLR computation."""
        batch_size, total_symbols = symbols.shape
        
        symbols_np = symbols.cpu().numpy()
        symbols_tf = tf.convert_to_tensor(symbols_np, dtype=tf.complex64)
        
        noise_power = 1.0 / (10 ** (snr / 10))
        
        demapper = Demapper(demapping_method="app", 
                            constellation_type="qam", 
                            num_bits_per_symbol=Mqam)
        
        llr_tf = demapper((symbols_tf, noise_power))
        llr_np = llr_tf.numpy()
        llr = torch.from_numpy(llr_np).to(symbols.device)
        
        if bits_per_pkg is None:
            bits_per_symbol = Mqam
            bits_per_pkg = (total_symbols * bits_per_symbol) // (self.toknum // self.toks_per_pkg)
        
        return llr.reshape(batch_size, -1, bits_per_pkg)
    
    @torch.no_grad()
    def awgn_channel(self, symbols, snr):
        """Add AWGN noise to symbols."""
        batch_size, total_symbols = symbols.shape
        noise_power = 1.0 / (10 ** (snr / 10))
        noise_real = torch.randn_like(symbols, dtype=torch.float32, device=symbols.device) * np.sqrt(noise_power / 2)
        noise_imag = torch.randn_like(symbols, dtype=torch.float32, device=symbols.device) * np.sqrt(noise_power / 2)
        noise = noise_real + 1j * noise_imag
        return symbols + noise
    
    def add_sionna_crc(self, bits):
        """Add CRC error detection."""
        batch_size, num_tokens, num_bits = bits.shape
        device = bits.device
        
        bits_np = bits.cpu().numpy()
        bits_tf = tf.convert_to_tensor(bits_np, dtype=tf.float32)
        
        encoded_bits_tf = self.crc_encoder(bits_tf)
        encoded_bits_np = encoded_bits_tf.numpy()
        encoded_bits = torch.tensor(encoded_bits_np, device=device)
        
        return encoded_bits
    
    def verify_sionna_crc(self, encoded_bits):
        """Verify CRC and detect errors."""
        batch_size, num_tokens, encoded_bits_len = encoded_bits.shape
        num_bits = encoded_bits_len - self.crc_degree
        device = encoded_bits.device
        
        encoded_bits_np = encoded_bits.cpu().numpy()
        encoded_bits_tf = tf.convert_to_tensor(encoded_bits_np, dtype=tf.float32)
        
        decoded_bits_tf, error_mask_tf = self.crc_decoder(encoded_bits_tf)
        
        decoded_bits_np = decoded_bits_tf.numpy()
        decoded_bits = torch.tensor(decoded_bits_np, device=device)
        
        error_mask_np = error_mask_tf.numpy()
        error_mask = torch.tensor(~error_mask_np, device=device, dtype=torch.bool)
        
        return decoded_bits, error_mask.reshape(batch_size, num_tokens)
    
    def add_conv_coding(self, bits):
        """Add convolutional coding."""
        batch_size, num_tokens, num_bits = bits.shape
        device = bits.device
        
        bits_np = bits.cpu().numpy()
        bits_tf = tf.convert_to_tensor(bits_np, dtype=tf.float32)
        
        encoded_bits_tf = self.conv_encoder(bits_tf)
        encoded_bits_np = encoded_bits_tf.numpy()
        encoded_bits = torch.tensor(encoded_bits_np, device=device)
        
        return encoded_bits
    
    def decode_conv(self, llr):
        """Decode convolutional code."""
        batch_size, num_tokens, encoded_bits_len = llr.shape
        device = llr.device
        
        llr_np = llr.cpu().numpy()
        llr_tf = tf.convert_to_tensor(llr_np, dtype=tf.float32)
        
        decoded_bits_tf = self.conv_decoder(llr_tf)
        decoded_bits_np = decoded_bits_tf.numpy()
        decoded_bits = torch.tensor(decoded_bits_np, device=device)
        
        return decoded_bits
    
    @torch.no_grad()
    def calculate_errors(self, tx_ids, rx_ids):
        """Calculate token and package error rates."""
        batch_size, num_tokens = tx_ids.shape
        num_packages = num_tokens // self.toks_per_pkg
        device = tx_ids.device
        
        # Token-level errors
        error_mask_token = (tx_ids != rx_ids).bool()
        token_level_ser = error_mask_token.float().mean().item()
        
        # Package-level errors
        error_mask_pkg = torch.zeros(batch_size, num_packages, device=device, dtype=torch.bool)
        b_idx, t_idx = error_mask_token.nonzero(as_tuple=True)
        pkg_idx = self.token_to_pkg_pos[t_idx, 0]
        error_mask_pkg[b_idx, pkg_idx] = True
        
        pkg_level_ser = error_mask_pkg.float().mean().item()
        
        # Create token mask from package errors
        token_pkg_map = self.token_to_pkg_pos[:, 0].unsqueeze(0).expand(batch_size, -1)
        error_mask_tok = error_mask_pkg.gather(1, token_pkg_map)
        
        return pkg_level_ser, token_level_ser, error_mask_tok
    
    @torch.no_grad()
    def simulate_transmission(self, tx_ids, snr=40, Mqam=4):
        """Simulate complete transmission pipeline."""
        # Package tokens
        tx_pkgs = self.tokens_to_pkgs_interleaved(tx_ids)
        tx_bits_raw = tx_pkgs.reshape(-1, self.bits_per_tok * self.toks_per_pkg)
        
        # Add error correction
        tx_pkgs = self.add_sionna_crc(tx_pkgs)
        tx_pkgs = self.add_conv_coding(tx_pkgs)
        
        # Modulate and transmit
        tx_symbols = self.QAM_modulation(tx_pkgs, Mqam)
        rx_symbols = self.awgn_channel(tx_symbols, snr)
        
        # Demodulate and decode
        rx_llr = self.QAM_demodulation_llr(
            rx_symbols, snr, Mqam, 
            bits_per_pkg=int((self.bits_per_tok * self.toks_per_pkg + self.crc_degree) / self.rate)
        )
        rx_bits = self.decode_conv(rx_llr)
        rx_bits, crc_status = self.verify_sionna_crc(rx_bits)
        
        # Convert back to tokens
        rx_ids = self.pkgs_to_tokens_interleaved(rx_bits)
        
        # Calculate errors
        bit_error_rate = (tx_bits_raw != rx_bits.reshape(tx_bits_raw.shape)).float().mean().item()
        pkg_level_ser, token_level_ser, error_mask_tok = self.calculate_errors(tx_ids, rx_ids)
        
        print(f"SNR: {snr} dB, PER: {pkg_level_ser:.6f}, TER: {token_level_ser:.6f}, BER: {bit_error_rate:.6f}")
        
        return pkg_level_ser, token_level_ser, bit_error_rate, rx_ids, rx_bits, error_mask_tok

# Initialize wireless simulation
wireless_sim = WirelessSimulation(toknum=256, toks_per_pkg=4, bits_per_tok=10, rate=0.5)
print("Wireless simulation initialized!")


## Evaluation Metrics and Models


In [ ]:
# Initialize evaluation models
loss_fn_alex = lpips.LPIPS(net='alex').to(config.device)
loss_fn_alex.eval()

CLIPmodel, preprocess = clip.load("ViT-B/32")
CLIPmodel = CLIPmodel.to(config.device)

def calculate_average(values):
    """Calculate average of a list of values."""
    return sum(values) / len(values)

print("Evaluation models loaded successfully!")


## Main Experiment: TokCom Performance Evaluation


In [ ]:
# Initialize results storage
results = []
total_range = config.patch_size * config.patch_size

print("Starting TokCom evaluation experiment...")
print(f"Testing {len(config.class_indices)} classes across {len(config.snr_list)} SNR levels")

for class_idx in tqdm(range(len(config.class_indices)), desc="Processing classes"):
    # Load images for current class
    image_paths = ImagePathAll[class_idx]
    class_label = ClassLabelAll[class_idx]
    
    # Prepare labels
    labels = class_label * torch.ones(config.num_images, dtype=torch.long).to(config.device)
    
    # Load and encode images
    images_orig = []
    orig_codes = []
    
    for img_path in image_paths:
        # Load and preprocess image
        x = preprocess_image(img_path, transform).to(config.device)
        images_orig.append(x)
        
        # Encode to tokens
        with torch.no_grad():
            emb, _, [_, _, code] = maskgit.ae.encode(x)
            code = code.reshape(x.size(0), config.patch_size, config.patch_size)
        orig_codes.append(code)
    
    # Stack images and codes
    images_orig = torch.cat(images_orig, dim=0)
    orig_codes_batch = torch.cat(orig_codes, dim=0)
    orig_codes = orig_codes_batch.clone().view(config.num_images, total_range)
    
    # Create output directory for this class
    save_path = os.path.join(config.save_dir, f"Sample_{class_label}")
    os.makedirs(save_path, exist_ok=True)
    
    # Test across different SNR levels
    for snr_idx, snr in enumerate(config.snr_list):
        # Initialize error counters
        PER_total = TER_total = BER_total = 0
        
        # Run multiple simulations for statistical significance
        for sim in range(config.num_simulations):
            # Simulate transmission
            per, ter, ber, rx_ids, rx_bits, error_mask_tok = wireless_sim.simulate_transmission(
                orig_codes, snr=snr, Mqam=4
            )
            
            PER_total += per
            TER_total += ter
            BER_total += ber
            
            # Direct decoding (baseline)
            with torch.no_grad():
                x_rx = maskgit.ae.decode_code(rx_ids.view(config.num_images, config.patch_size, config.patch_size))
            
            # TokCom reconstruction
            mask = torch.zeros_like(orig_codes, device=config.device)
            masked_codes = orig_codes.clone()
            
            if error_mask_tok.sum().item() != 0:
                mask[error_mask_tok] = 1
                masked_codes[error_mask_tok] = config.mask_value
            
            # Generate reconstruction using MaskGiT
            gen_sample, gen_orig, _, _, _, _, _ = maskgit.GenDecoding(
                init_code=masked_codes.view(config.num_images, config.patch_size, config.patch_size).clone(),
                _mask=mask.view(config.num_images, config.patch_size, config.patch_size).clone(),
                orig_code=orig_codes_batch,
                nb_sample=config.num_images,
                labels=labels,
                sm_temp=config.sm_temp,
                w=1,  # Classifier-free guidance weight
                randomize=config.randomize,
                r_temp=config.r_temp,
                sched_mode=config.sched_mode,
                step=config.step
            )
            
            # Calculate metrics for TokCom reconstruction
            psnr_values = [calculate_psnr(img1, img2) for img1, img2 in zip(images_orig, gen_sample)]
            lpips_values = [calculate_lpips(img1.unsqueeze(0), img2.unsqueeze(0), loss_fn_alex) 
                           for img1, img2 in zip(images_orig, gen_sample)]
            clip_scores = [calculate_clip_score(img1, img2, CLIPmodel, preprocess) 
                          for img1, img2 in zip(images_orig, gen_sample)]
            
            # Calculate metrics for direct decoding
            psnr_values_rx = [calculate_psnr(img1, img2) for img1, img2 in zip(images_orig, x_rx)]
            lpips_values_rx = [calculate_lpips(img1.unsqueeze(0), img2.unsqueeze(0), loss_fn_alex) 
                              for img1, img2 in zip(images_orig, x_rx)]
            clip_scores_rx = [calculate_clip_score(img1, img2, CLIPmodel, preprocess) 
                             for img1, img2 in zip(images_orig, x_rx)]
            
            # Calculate reference metrics (perfect reconstruction)
            if snr_idx == 0 and sim == 0:
                psnr_values_ref = [calculate_psnr(img1, img2) for img1, img2 in zip(images_orig, gen_orig)]
                lpips_values_ref = [calculate_lpips(img1.unsqueeze(0), img2.unsqueeze(0), loss_fn_alex) 
                                  for img1, img2 in zip(images_orig, gen_orig)]
                clip_scores_ref = [calculate_clip_score(img1, img2, CLIPmodel, preprocess) 
                                 for img1, img2 in zip(images_orig, gen_orig)]
                
                avg_psnr_ref = calculate_average(psnr_values_ref)
                avg_lpips_ref = calculate_average(lpips_values_ref)
                avg_clip_ref = calculate_average(clip_scores_ref)

            if sim == 0:  
                # Visualize one example: Reference vs Direct Decoding vs TokCom
                idx_to_show = 0  # show the first image in the batch
                fig, axs = plt.subplots(1, 3, figsize=(12, 4))
                show_items = [
                    ("Reference", gen_orig[idx_to_show].detach().cpu()),
                    ("Direct Decoding", x_rx[idx_to_show].detach().cpu()),
                    ("TokCom", gen_sample[idx_to_show].detach().cpu()),
                ]
                for ax, (title, img_t) in zip(axs, show_items):
                    ax.imshow(img_t.permute(1, 2, 0).clamp(0, 1).numpy())
                    ax.set_title(title)
                    ax.axis('off')
                plt.tight_layout()
                plt.show()
            
            # Calculate average metrics
            avg_psnr = calculate_average(psnr_values)
            avg_lpips = calculate_average(lpips_values)
            avg_clip = calculate_average(clip_scores)
            
            avg_psnr_rx = calculate_average(psnr_values_rx)
            avg_lpips_rx = calculate_average(lpips_values_rx)
            avg_clip_rx = calculate_average(clip_scores_rx)
            
            # Store results
            result = {
                'class_label': class_label,
                'snr': snr,
                'PER': per,
                'TER': ter,
                'BER': ber,
                'avg_PSNR': avg_psnr,
                'avg_CLIP': avg_clip,
                'avg_LPIPS': avg_lpips,
                'avg_PSNR_rx': avg_psnr_rx,
                'avg_CLIP_rx': avg_clip_rx,
                'avg_LPIPS_rx': avg_lpips_rx,
                'avg_PSNR_ref': avg_psnr_ref,
                'avg_CLIP_ref': avg_clip_ref,
                'avg_LPIPS_ref': avg_lpips_ref
            }
            results.append(result)
            
            # Print results
            print(f"Class: {class_label}, SNR: {snr} dB | "
                  f"PSNR: {avg_psnr:.2f}(TokCom)/{avg_psnr_rx:.2f}(Direct)/{avg_psnr_ref:.2f}(Ref) | "
                  f"CLIP: {avg_clip:.4f}(TokCom)/{avg_clip_rx:.4f}(Direct)/{avg_clip_ref:.4f}(Ref) | "
                  f"LPIPS: {avg_lpips:.4f}(TokCom)/{avg_lpips_rx:.4f}(Direct)/{avg_lpips_ref:.4f}(Ref)")
        
        # Print average results for this SNR
        print(f"\nClass {class_label} | SNR: {snr} dB | "
              f"PER: {PER_total/config.num_simulations:.6f}, "
              f"TER: {TER_total/config.num_simulations:.6f}, "
              f"BER: {BER_total/config.num_simulations:.6f}\n")

print("\nExperiment completed! Saving results...")

# Save results to file
with open('tokcom_results.txt', 'w') as f:
    for result in results:
        f.write(f"Class: {result['class_label']}\n")
        f.write(f"SNR: {result['snr']}\n")
        f.write(f"PER: {result['PER']}\n")
        f.write(f"TER: {result['TER']}\n")
        f.write(f"BER: {result['BER']}\n")
        f.write(f"PSNR_TokCom: {result['avg_PSNR']}\n")
        f.write(f"PSNR_Direct: {result['avg_PSNR_rx']}\n")
        f.write(f"PSNR_Ref: {result['avg_PSNR_ref']}\n")
        f.write(f"CLIP_TokCom: {result['avg_CLIP']}\n")
        f.write(f"CLIP_Direct: {result['avg_CLIP_rx']}\n")
        f.write(f"CLIP_Ref: {result['avg_CLIP_ref']}\n")
        f.write(f"LPIPS_TokCom: {result['avg_LPIPS']}\n")
        f.write(f"LPIPS_Direct: {result['avg_LPIPS_rx']}\n")
        f.write(f"LPIPS_Ref: {result['avg_LPIPS_ref']}\n\n")

print("Results saved to 'tokcom_results.txt'")


## Summary

This notebook demonstrates the TokCom framework for robust image transmission over wireless channels. Key features:

1. **Token-based Communication**: Images are encoded into discrete tokens using VQGAN
2. **Error Correction**: CRC and convolutional coding provide robustness against channel errors
3. **Generative Reconstruction**: MaskGiT reconstructs images from corrupted tokens
4. **Performance Evaluation**: Comprehensive metrics including PSNR, CLIP, and LPIPS

The results show that TokCom significantly outperforms direct decoding, especially at low SNR levels, demonstrating the effectiveness of generative reconstruction for semantic communication.

For more details, please refer to the original paper: "Token Communications: A Large Model-Driven Framework for Cross-Modal Context-Aware Semantic Communications"
